# Step 21.5 — Context cropping：完整历史留下，每轮只读末尾窗口

先在仓库根目录同步母版：

```bash
git pull
```

再用时间戳与 `cp -n` 建立独立工作副本；仓库已有 `notebooks/work/`，同名文件不会覆盖：

```bash
cp -n notebooks/96_step21_5_context_cropping.ipynb notebooks/work/96_step21_5_context_cropping_$(date +%Y%m%d_%H%M%S)_work.ipynb
```

以上命令供未来使用，本次未执行。沿用 Python 3.12 / .venv，API 固定 **MLX v0.32.2**。

本节只改变 21.1 的“每轮给模型哪些输入”，保留原来的 greedy 选择与 append。先读三轮纸算和两处关键动作，再看后面的完整源码；复用依赖与新循环分开标明，不需要旧 kernel 变量。

2026-11-17 教学修订：**21.5 累计 1/2＝历史修正 0/2＋本次教学修订 1 次**。原函数、人工表和未来调用的代码保持不变，仅分组并重写讲解；练习限定在窗口管理。全部代码单元 execution_count=null、outputs=[]。本次没有运行课程代码、采样、训练、测试或实验，没有语料/checkpoint 操作，也未操作用户工作副本。

## 1. 一条序列，两个不同的用途

[21.1 的完整循环](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/bc27834361392b47d0b007277769ab3206948e1c/notebooks/92_step21_1_generation_loop.ipynb)把整条已有前缀交给模型。它预先检查最后一次 forward 的长度 `L+n−1`，超过 block_size 就拒绝。现在希望输出可以继续增长，同时每次 forward 都不超过约定窗口 W。

为此分开保存两个对象：

| 名称 | 用途 | 第 r 轮开始时的 shape |
|---|---|---|
| **history（完整历史）** | 留下原 prompt 和已经生成的全部 token，最终返回给调用者 | (1,L₀+r) |
| **context / window（本轮上下文窗口）** | 只取 history 最后至多 W 个 token，作为本轮模型输入 | (1,Tᵣ)，Tᵣ=min(L₀+r,W) |

L₀ 是非空 prompt 长度，r 从 0 开始，表示已经追加的 token 数；本节 B=1。每轮只增加一个 ID，所以 history 的长度持续增长，而 context 的长度到 W 后不再增长。

**裁剪输入窗口，不裁剪返回的历史。** 每轮先从 history 取 context，选出一个 ID 后仍追加到完整 history。若反过来用裁短的 context 替换 history，再追加，最早的 prompt 就会从返回结果中丢掉。限制模型可直接读取的范围不等于从输出文本删除旧内容，也不自动赋予模型长上下文能力。

## 2. 负切片为什么只取末尾，却不删除 history？

关键表达式是 `history[:, -block_size:]`，其中 W=block_size：

- 第一个 `:` 保留 batch 轴；不是把 batch 展平。
- `-W:` 从末尾向前数 W 项开始，取到时间轴末尾。长度 L 不足 W 时从现有开头取，不补零、不补 token。
- 用从 0 开始的索引表示，实际保留区间是 `[max(0,L−W), L)`；右端 L 不包含在内，长度就是 min(L,W)。

例如 L=2、W=3 时保留索引 [0,2)，长度 2；L=4、W=3 时保留 [1,4)，长度 3。这里计算的是 token 数，不是字符串字节数。

`context = ...` 只把切片结果赋给另一个名字，没有把 history 重新赋值成切片。本算法也不会修改 context 的元素，所以不依赖“切片是否共享底层存储”来保留历史。W 必须为正整数：`-0` 等于 0，`[:, -0:]` 实际会保留全长，不能用它表示零窗口；原函数会在循环前拒绝 W=0。

一轮的形状链仍沿用 21.1，只是 forward 的时间轴换成 Tᵣ：

```text
history (1,L₀+r) → context (1,Tᵣ) → logits (1,Tᵣ,V)
→ last (1,V) → argmax ID (1,) → token (1,1)
→ append 到 history → 新 history (1,L₀+r+1)
```

## 3. 先在纸上走三轮：窗口饱和，历史仍增长

沿用 21.1 的人工四字符表：`A=0, B=1, C=2, _=3`。这个 toy 在最后位置只根据当前 token 给分，表中 A 行选择 B、B 行选择 C、C 行选择 A；各行的最大值都唯一。这些是指定规则，不是学习结果。

令 prompt=[A,B]、W=3、只考察前三轮。V=4，因此 last 始终是 (1,4)，选出的 token 增轴后始终是 (1,1)：

| r | 调用前完整 history / shape | 送模型的 context / shape | logits shape | 最后分数 → 选中 ID | 追加后的完整 history / shape |
|---:|---|---|---|---|---|
| 0 | [A,B] / (1,2) | [A,B] / (1,2) | (1,2,4) | B 行 [0,0,3,−1] → 2=C | [A,B,C] / (1,3) |
| 1 | [A,B,C] / (1,3) | [A,B,C] / (1,3) | (1,3,4) | C 行 [3,0,0,−1] → 0=A | [A,B,C,A] / (1,4) |
| 2 | [A,B,C,A] / (1,4) | [B,C,A] / (1,3) | (1,3,4) | A 行 [0,3,1,−1] → 1=B | [A,B,C,A,B] / (1,5) |

第三轮送入模型的是 history 的索引 1、2、3；原索引 0 的 A 仍在 history 中，只是不在这次 forward 的输入里。返回的历史长度为 5，而三次输入长度记录为 [2,3,3]。最后追加的 B 在这三轮中尚未再次送入模型。

以上逐项由指定表和切片规则推导，**不是执行后打印的结果**。它展示输入/输出管理，不能用来判断真实 Transformer 裁剪前后的预测是否相同。

### 3.1 这个 toy 不能证明“裁剪不影响预测”

原 toy 的 `toy_logits(ids)=TRANSITION[ids]` 在每个位置只查当前 token。对任何非空窗口，裁剪都保留最后一个 token，所以这个 toy 的最后分数不因丢掉更早 token 而改变。这是人工表的限制，不是 Transformer 的性质。

可以用一个**仅在纸上定义、没有新增代码**的上下文敏感规则看出区别：若输入中含 `_`，最后分数指定为 [0,0,4,−1]（选 C）；否则指定为 [0,4,0,−1]（选 B）。

| 给这个纸上规则的完整历史 | 不裁剪时的选择 | W=3 后实际输入 | 裁剪后的选择 |
|---|---|---|---|
| [A,B,C,A] | B | [B,C,A] | B |
| [_,B,C,A] | C | [B,C,A] | B |

两条完整历史不同，裁剪后的输入相同，因而这个规则无法再根据被裁掉的 `_` 区分它们。这里不是模拟 attention 或宣称真实模型必定怎样预测，只说明“最后 token 一样”不足以保证任意上下文敏感模型的分数不变。

对本节无 cache、无额外可变状态的回调，在固定参数及相同模式下，本轮**直接访问**的是传入的 context。更老的 token 仍可能曾影响已经生成的 token，而这些后续 token 又留在当前窗口里，形成间接影响。因此不能说旧 token 从此“完全没有影响”；也不能反过来说窗口已经无损保存了全部旧信息。若未来回调暗中读取外部历史或 cache，就已经超出这里的接口假设。

## 4. 只看本节新增的动作

### 4.1 先取窗口，再记录这次真正送入的长度

下面是原函数中的三行摘录，供逐行阅读；完整定义在第 6 节：

```text
context = history[:, -block_size:]
lengths.append(context.shape[1])
last = last_logits(logits_fn, context, vocab_size)
```

每轮都从**最新 history**重新切片。`lengths` 是 Python list，`append` 向它添加本次 context 的时间轴长度；它不是 token 列表，也不会参与模型预测。把记录放在追加 token 之前，才能与这次 forward 一一对应。

最关键的参数变化是把 `context` 交给 last_logits。若仍传 history，虽然创建了一个切片变量，模型却依旧读全长，窗口限制没有生效。last_logits 本身不裁剪，它仍只做调用回调、核对 (B,T,V)、取末行和有限值检查。

### 4.2 选择规则复用，追加目标仍是完整历史

```text
token = mx.argmax(last, axis=-1).astype(mx.int32)[:, None]
history = mx.concatenate([history, token], axis=1)
mx.eval(history)
```

`argmax`、整数 dtype、`[:,None]` 和 time 轴拼接均已在 21.1 解释；这里仍是 greedy，没有温度、top-k 或随机 key。21.2–21.4 的选择器是另一处可替换环节，本轮不把它们混入窗口管理函数。

注意 concatenate 左侧输入是 **history**。写成 `[context,token]` 会丢弃旧输出；写对 shape 还不够，必须保留正确的序列对象。求值边界 `mx.eval(history)` 完成本轮新序列及其依赖，不更新权重，也不等于 `model.eval()`。

循环结束返回 `(history,lengths)`：前者是 MLX 整数数组，后者是 Python 长度列表。调用者要分别接收两个返回值，不能把整个 tuple 当成下一轮 prompt 或模型输入。

## 5. 与 21.1 相比，究竟改了哪里？

| 21.1 generate_greedy | 本节 generate_cropped | 原因 |
|---|---|---|
| 预检 L+n−1 超过 block_size 就拒绝 | 不再用这个总长度条件拒绝 | 每次只送最多 W 项，完整输出允许继续增长 |
| ids 保存完整前缀，并直接送入模型 | history 保存完整前缀；新增 context 切片送模型 | 区分输出历史与当次输入 |
| 没有输入长度记录 | 新增 lengths，并在每次 forward 前 append | 让调用者能对照输入长度与输出长度 |
| argmax → 增轴 → 拼接 → eval | 同一组选择/追加/求值动作 | 不改变 greedy 策略或参数 |
| 返回 ids | 返回 history,lengths | 明确新增的记录接口 |

`ids→history` 与 `nxt→token` 只是命名变化。真正的行为变化是去掉全前缀容量拒绝、按轮截取模型输入，以及增加记录返回值。`check_prompt` 和 `last_logits` 的定义继续复用；原函数对非空合法 prompt、正整数 W、非负整数 n 的保护仍保留。

**边界也按当前源码解释。** n=0 时仍检查 prompt/W/n，返回完整 prompt 和空 lengths，不调用回调；即使 prompt 长于 W，也没有 forward 需要裁剪。n>0 且 prompt 已长于 W 时，第一轮就读末尾 W 项，返回历史仍含全部 prompt。

block_size 在这里是调用者指定的窗口上限；此 helper 不会自动检查真实模型的 max_seq_len。真实模型的容量和模式需要在接入前匹配（见可选展望），不能只把 W 改大来绕过模型自身限制。

## 6. 完整源码：复用依赖与新循环分开

以下三个代码单元合起来就是原来的完整定义，计算顺序、函数签名、保护条件和人工分数未变；只是将集中代码按用途分组。所有依赖都在本页，不必执行旧 Notebook。

### 6.1 已学 toy 依赖：只为窗口流程提供分数

沿用 [21.1 第 4 节](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/bc27834361392b47d0b007277769ab3206948e1c/notebooks/92_step21_1_generation_loop.ipynb)的人工表、词表和回调。`TRANSITION[ids]` 为每个 ID 查一行，输入 (B,T)，输出 (B,T,4)。它既不训练，也不读取语料或真实权重。

第一次阅读先确认这三个 shape 即可，不需要把已讲的查表重新当作本节的新算法。下面仍是未执行源码，包括创建人工数组的语句也没有被运行。

In [ ]:
import math
import mlx.core as mx

# 本单元及后续所有单元均为未执行的教学源码。
# 仅在未来独立工作副本中按顺序运行；本次材料制作没有导入 MLX。

# 人工定义转移分数；不是语料训练结果。
TOY_VOCAB = ("A", "B", "C", "_")
TRANSITION = mx.array([[0., 3., 1., -1.],
                       [0., 0., 3., -1.],
                       [3., 0., 0., -1.],
                       [3., 1., 0., -1.]], dtype=mx.float32)

def toy_logits(ids):
    # 输入 int32 (B,T)，合法 ID 0..3；输出 (B,T,4)。
    # 这个模型只看当前 token，故意不模拟真实 Transformer。
    return TRANSITION[ids]



### 6.2 已学保护性检查：不承担裁剪职责

这两个 helper 与 21.1 的函数定义一致：

- `check_prompt` 接受非空 list/tuple，要求每项为合法 Python 整数 ID；布尔值不作为 ID 接受。
- `last_logits` 要求回调返回 `(*ids.shape,V)`，再取 `[:,−1,:]` 转成 float32，求值并检查末行有限性。这里的 ids 形参会收到本轮 context；它只检查用于选择的末行，不是全模型健康检查。

`.item()` 把归约结果交给 Python 条件判断，属于 host 检查；本节没有把它放入 mx.compile。保留它是为了在 argmax 前拒绝不合法结果，不是为了增加新的生成策略。

In [ ]:
def check_prompt(prompt, vocab_size):
    if not isinstance(prompt, (list, tuple)) or not prompt:
        raise ValueError("prompt must be a nonempty list/tuple of IDs")
    if any(type(t) is not int or not 0 <= t < vocab_size for t in prompt):
        raise ValueError("prompt contains an invalid token ID")

def last_logits(logits_fn, ids, vocab_size):
    logits = logits_fn(ids)
    if logits.shape != (*ids.shape, vocab_size):
        raise ValueError("callback must return (B,T,V) logits")
    last = logits[:, -1, :].astype(mx.float32)
    mx.eval(last)
    if not bool(mx.all(mx.isfinite(last)).item()):
        raise ValueError("model logits must be finite")
    return last



### 6.3 本节完整函数：按第 4–5 节逐行对照

`*` 后的 vocab_size 与 block_size 必须按名字传入。循环内 `_` 只是未使用的轮次变量，实际序列状态由 history 保存。循环前初始化 lengths，循环后返回完整历史和输入长度记录；没有读取 checkpoint、重设模型或隐藏的采样入口。

In [ ]:
def generate_cropped(logits_fn, prompt, max_new_tokens, *, vocab_size, block_size):
    check_prompt(prompt, vocab_size)
    if type(block_size) is not int or block_size <= 0:
        raise ValueError("block_size must be a positive integer")
    if type(max_new_tokens) is not int or max_new_tokens < 0:
        raise ValueError("max_new_tokens must be a nonnegative integer")
    history = mx.array([prompt], dtype=mx.int32)
    lengths = []
    for _ in range(max_new_tokens):
        context = history[:, -block_size:]
        lengths.append(context.shape[1])
        last = last_logits(logits_fn, context, vocab_size)
        token = mx.argmax(last, axis=-1).astype(mx.int32)[:, None]
        history = mx.concatenate([history, token], axis=1)
        mx.eval(history)
    return history, lengths

## 7. 可选未来观察：两种“长度”会在哪一步分开？

下面保留原来的调用：prompt=[0,1]、W=3，追加 **5** 个 token。第 3 节只手算了它的前三轮；未来可以先补完后两轮，再决定是否在独立工作副本执行。本次没有调用它，也没有预填 print 的输出。

观察问题是：history 每次追加后增长，但 context_lengths 是否会在窗口饱和后固定？若不符合，先检查真正传给 last_logits 的变量，以及拼接的是 history 还是 context，而不是先怀疑模型参数。

`context_lengths` 记录每次 forward 前的长度；`history.tolist()` 显示完整返回序列。这个观察能检查窗口和返回值管理，不能证明真实 Transformer 的预测等价、长文质量或 cache 正确性。这里也没有随机采样，不能把它包装成分布验证。

In [ ]:
history, context_lengths = generate_cropped(
    toy_logits, [0, 1], 5, vocab_size=4, block_size=3)
mx.eval(history)
print("forward context lengths:", context_lengths)
print("full output:", history.tolist())

## 8. 为什么相邻两轮仍然重复计算？

第 3 节第二轮输入 [A,B,C]，第三轮输入 [B,C,A]。虽然 B、C 重叠，函数每轮都再次调用 `logits_fn(context)`，没有保存模型内部结果供下一轮使用。history 只保存 **token ID**，lengths 只保存 **长度**，两者都不是 KV cache。

换成真实 MiniGPT 时，每次调用会重做整个窗口的 embedding、投影、attention 和 MLP。以一层、一个 attention head、head_dim=d 为范围，长度 T 的 attention 分数和加权求和主项随 T²d 增长；N 轮为约 `O(Σᵣ Tᵣ²d)`。窗口都已达到 W 时可简写为 `O(NW²d)`。O 记号表示随规模增长的工作量级，忽略常数；这里也未计入其他层/头、投影和 MLP 成本。

这不是秒数、峰值内存或实测速度。裁剪限制每次模型输入长度，但本函数仍保留并逐轮拼接完整 history，不能因此声称整个生成过程的内存或所有开销都固定不变。

## 9. 可选展望：真实位置表与 KV cache

这部分解释当前方案的边界，不要求在本节实现 cache 或回答其工程练习。

已有 [15.4 MiniGPT](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/bc27834361392b47d0b007277769ab3206948e1c/notebooks/65_step15_4_implement_minigpt.ipynb)在每次 forward 中构造 `position_ids=mx.arange(T)`，再查 `position_embedding`。因此窗口内位置每次从 0 开始：在 [A,B,C] 中 B 位于 1，在下一轮 [B,C,A] 中 B 位于 0。这是 **learned absolute positions（学习的绝对位置）**下重新编号的窗口输入，不是保留整条 history 的全局位置。

未来接真实模型时，应保持 W≤model.max_seq_len，并核对训练时约定的有效上下文、真实词表和模型模式。能让完整输出超过 W，不等于学到了更大位置或可靠的长距离依赖；本页不加载任何真实模型或 checkpoint。

**KV cache（键值缓存）**尝试复用之前算过的 K/V，而本循环完全重算。不能简单给当前窗口附上旧 cache 就宣称等价：窗口滑动会改变 learned positions；深层旧表示也可能包含已离开窗口的前缀信息，旧 K/V 不一定等于重新计算当前窗口的 K/V。

未来若讨论 offset（位置偏移，给位置编号加起点）、RoPE 或其他窗口/cache 策略，还必须一起约定位置、可见范围与缓存生命周期。本节没有这些实现；即使采用 offset，也不能把它当作“旧 cache 必然有效”的证明。

## 10. 自己检查理解（不附答案）

1. W=4、prompt 长 6、生成 3 个 token 时，用从 0 开始的索引列出每轮窗口区间、forward shape 和最终返回长度。不必猜模型会选择哪个具体 ID。
2. 用文字说明：把 `history = concatenate([history,token],axis=1)` 误改为拼接 context，会破坏哪条返回值约定？“shape 合法”为什么不足以证明它正确？
3. 对 n=0 和 n>0 两种情况，分别解释 lengths 中一项对应哪个时刻、是否有模型调用；再说明 lengths 为什么不能充当生成的 token 序列。

练习只要求窗口、历史和接口推理，不要求画 KV cache、设计 offset 或运行真实模型。请先写自己的判断；母版不附练习答案。

## 11. 固定版本依据与材料状态

- [MLX v0.32.2 indexing guide](https://github.com/ml-explore/mlx/blob/v0.32.2/docs/src/usage/indexing.rst)：负索引、切片、多轴与 None 新增轴。
- [MLX v0.32.2 ops.cpp](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/ops.cpp)：argmax、concatenate、isfinite/all。
- [MLX v0.32.2 transforms.cpp](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/transforms.cpp)：显式求值接口。
- [21.1 已验收循环](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/bc27834361392b47d0b007277769ab3206948e1c/notebooks/92_step21_1_generation_loop.ipynb)与[真实 MiniGPT 位置接口](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/bc27834361392b47d0b007277769ab3206948e1c/notebooks/65_step15_4_implement_minigpt.ipynb)是本节的课程衔接依据。

本次重写教学组织，修正“旧 token 完全不再影响”这一过强说法；保留原来的 API、检查条件、计算及未来调用，源码只分组，不添加新可执行入口。三轮表和上下文敏感反例均为纸面说明。

仅做原始 JSON/schema、合法唯一 ID、静态语法/AST、源码一致性和文件范围检查；所有单元未执行、输出为空。没有导入 MLX、运行示例、训练、测试、实验、语料读取/下载或 checkpoint/work-copy 操作，也没有真实结果或性能结论。

21.5 累计 **1/2**（历史 0/2＋本次教学修订 1 次），其他历史计数保持。本轮停在 21.5，等待独立验收；材料齐备不等于学员掌握或运行验证通过。